In [1]:
# ============================================================
# CELL 1 — IMPORT
# ============================================================

import pandas as pd
import numpy as np
import re

from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Alignment

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ============================================================
# CELL 2 — FILE PATH
# ============================================================

BASE_DIR = Path(
    r"C:\Users\zerox\OneDrive\Documents\Desktop\learning"
)

# ------------------------------------------------------------
# MASTER TEMPLATE
# ------------------------------------------------------------

MASTER_FILE = (
    BASE_DIR
    / "data"
    / "master"
    / "Learning agility self assesment & superior_24.7.2026.xlsx"
)

# ------------------------------------------------------------
# RAW ASSESSMENT
# ------------------------------------------------------------

SELF_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Self Assessment MASTER Questions Answers.xlsx"
)

SUPERIOR_FILE = (
    BASE_DIR
    / "data"
    / "raw"
    / "Superior Assessment MASTER Questions Answers.xlsx"
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed"
    / "master_output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("========================================")
print("PATH CONFIGURATION")
print("========================================")

print("Master   :", MASTER_FILE)
print("Exists   :", MASTER_FILE.exists())

print("\nSelf     :", SELF_FILE)
print("Exists   :", SELF_FILE.exists())

print("\nSuperior :", SUPERIOR_FILE)
print("Exists   :", SUPERIOR_FILE.exists())

print("\nOutput   :", OUTPUT_DIR)
print("Exists   :", OUTPUT_DIR.exists())

PATH CONFIGURATION
Master   : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\master\Learning agility self assesment & superior_24.7.2026.xlsx
Exists   : True

Self     : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\raw\Self Assessment MASTER Questions Answers.xlsx
Exists   : True

Superior : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\raw\Superior Assessment MASTER Questions Answers.xlsx
Exists   : True

Output   : C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\processed\master_output
Exists   : True


In [3]:
# ============================================================
# CELL 3 — LOAD MASTER + RAW
# ============================================================

# ------------------------------------------------------------
# LOAD MASTER
# ------------------------------------------------------------

master_wb = load_workbook(
    MASTER_FILE,
    data_only=False
)

# ------------------------------------------------------------
# LOAD RAW SELF
# ------------------------------------------------------------

df_self = pd.read_excel(
    SELF_FILE,
    engine="openpyxl"
)

# ------------------------------------------------------------
# LOAD RAW SUPERIOR
# ------------------------------------------------------------

df_superior = pd.read_excel(
    SUPERIOR_FILE,
    engine="openpyxl"
)

print("========================================")
print("FILES LOADED")
print("========================================")

print("Self     :", df_self.shape)
print("Superior :", df_superior.shape)

print("\nMaster sheets:")
for sheet in master_wb.sheetnames:
    print("-", sheet)

c:\Users\zerox\OneDrive\Documents\Desktop\learning\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


FILES LOADED
Self     : (32, 110)
Superior : (30, 109)

Master sheets:
- Identitas
- mental agility
- people agility
- change agility
- result agility
- self awareness
- Profile Summary
- Kompetensi
- sheet template
- Dasboard
- Aktivitas pengembangan
- Bukti Aktivitas Pengembangan
- Penilaian Perilaku
- Form Penilaian Presentasi
- panduan


In [4]:
# ============================================================
# CELL 4 — TEXT NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    value = str(value)

    # Non-breaking space
    value = value.replace("\xa0", " ")

    # Line breaks
    value = value.replace("\n", " ")
    value = value.replace("\r", " ")

    # Multiple spaces
    value = re.sub(r"\s+", " ", value)

    return value.strip().lower()

In [5]:
# ============================================================
# CELL 5 — EXTRACT MASTER 5 DIMENSIONS
# ============================================================

DIMENSIONS = {
    "mental_agility": "mental agility",
    "people_agility": "people agility",
    "change_agility": "change agility",
    "result_agility": "result agility",
    "self_awareness": "self awareness",
}

indicator_master = []

for dimension_key, sheet_name in DIMENSIONS.items():

    ws = master_wb[sheet_name]

    # Master memiliki 10 indikator:
    # Row 13 = indikator 1
    # Row 14 = indikator 2
    # ...
    # Row 22 = indikator 10

    for indicator_no, row in enumerate(range(13, 23), start=1):

        indicator = ws.cell(row, 2).value
        question = ws.cell(row, 3).value

        levels = {
            1: ws.cell(row, 4).value,
            2: ws.cell(row, 5).value,
            3: ws.cell(row, 6).value,
            4: ws.cell(row, 7).value,
            5: ws.cell(row, 8).value,
        }

        indicator_master.append({

            # ------------------------------------------------
            # IDENTITAS
            # ------------------------------------------------

            "dimension": dimension_key,

            "sheet_name": sheet_name,

            "indicator_no": indicator_no,

            "indicator_id": (
                f"{dimension_key}_{indicator_no:02d}"
            ),

            # ------------------------------------------------
            # INDICATOR
            # ------------------------------------------------

            "indicator": indicator,

            # ------------------------------------------------
            # QUESTION
            # ------------------------------------------------

            "question": question,

            "question_normalized": normalize_text(question),

            # ------------------------------------------------
            # LEVEL 1 - 5
            # ------------------------------------------------

            "level_1": levels[1],
            "level_2": levels[2],
            "level_3": levels[3],
            "level_4": levels[4],
            "level_5": levels[5],

            # ------------------------------------------------
            # NORMALIZED ANSWERS
            # ------------------------------------------------

            "level_1_normalized": normalize_text(levels[1]),
            "level_2_normalized": normalize_text(levels[2]),
            "level_3_normalized": normalize_text(levels[3]),
            "level_4_normalized": normalize_text(levels[4]),
            "level_5_normalized": normalize_text(levels[5]),
        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

indicator_master_df = pd.DataFrame(indicator_master)


# ============================================================
# DISPLAY STRUCTURE
# ============================================================

print("========================================")
print("MASTER STRUCTURE")
print("========================================")

print(
    "Total indicators :",
    len(indicator_master_df)
)

print("\nIndicators per dimension:")

print(
    indicator_master_df
    .groupby("dimension")
    .size()
)

print("\nIndicator IDs:")

print(
    indicator_master_df[
        [
            "dimension",
            "indicator_no",
            "indicator_id",
            "indicator"
        ]
    ].to_string(index=False)
)

MASTER STRUCTURE
Total indicators : 50

Indicators per dimension:
dimension
change_agility    10
mental_agility    10
people_agility    10
result_agility    10
self_awareness    10
dtype: int64

Indicator IDs:
     dimension  indicator_no      indicator_id                                                                                     indicator
mental_agility             1 mental_agility_01                                            Analytical Thinking\n(memahami data & fakta dasar)
mental_agility             2 mental_agility_02                                               Root Cause Analysis\n(mencari penyebab masalah)
mental_agility             3 mental_agility_03                          Critical Thinking & Inquisitiveness\n(mempertanyakan & mengevaluasi)
mental_agility             4 mental_agility_04                            Explaining Complexity\n(menjelaskan hal kompleks secara sederhana)
mental_agility             5 mental_agility_05                                    Int

In [6]:
# ============================================================
# CELL 6 — VALIDATE MASTER
# ============================================================

assert len(indicator_master_df) == 50, (
    f"Expected 50 indicators, "
    f"found {len(indicator_master_df)}"
)

assert (
    indicator_master_df["indicator_id"]
    .nunique()
    == 50
), "Duplicate indicator_id detected."

assert (
    indicator_master_df["question_normalized"]
    .nunique()
    == 50
), "Duplicate questions detected."

for col in [
    "level_1",
    "level_2",
    "level_3",
    "level_4",
    "level_5"
]:

    missing = (
        indicator_master_df[col]
        .isna()
        .sum()
    )

    assert missing == 0, (
        f"{col} has {missing} missing values."
    )


print("========================================")
print("MASTER VALIDATION")
print("========================================")
print("50 indicators       : OK")
print("50 unique questions : OK")
print("Level 1-5 complete  : OK")

MASTER VALIDATION
50 indicators       : OK
50 unique questions : OK
Level 1-5 complete  : OK


In [7]:
# ============================================================
# CELL 7 — DETECT RAW QUESTION COLUMNS
# ============================================================

def detect_question_columns(
    df,
    master_questions
):

    master_question_set = {
        normalize_text(q)
        for q in master_questions
    }

    question_columns = []

    for col in df.columns:

        normalized_col = normalize_text(col)

        if normalized_col in master_question_set:

            question_columns.append(col)

    return question_columns


master_questions = (
    indicator_master_df["question"]
    .tolist()
)

self_question_cols = detect_question_columns(
    df_self,
    master_questions
)

superior_question_cols = detect_question_columns(
    df_superior,
    master_questions
)

print("========================================")
print("RAW QUESTION VALIDATION")
print("========================================")

print(
    "Self questions detected     :",
    len(self_question_cols)
)

print(
    "Superior questions detected :",
    len(superior_question_cols)
)

assert len(self_question_cols) == 50, (
    "Self raw does not contain exactly "
    "50 Master questions."
)

assert len(superior_question_cols) == 50, (
    "Superior raw does not contain exactly "
    "50 Master questions."
)

print("\nQuestion structure: OK")

RAW QUESTION VALIDATION
Self questions detected     : 50
Superior questions detected : 50

Question structure: OK


In [8]:
# ============================================================
# CELL 8 — MASTER ANSWER → RATING MAPPING
# ============================================================

answer_mapping = {}

for _, row in indicator_master_df.iterrows():

    question_key = row["question_normalized"]

    answer_mapping[question_key] = {

        normalize_text(row["level_1"]): 1,

        normalize_text(row["level_2"]): 2,

        normalize_text(row["level_3"]): 3,

        normalize_text(row["level_4"]): 4,

        normalize_text(row["level_5"]): 5,
    }


print("========================================")
print("ANSWER MAPPING")
print("========================================")

print(
    "Questions mapped :",
    len(answer_mapping)
)

print(
    "Answers mapped   :",
    sum(
        len(v)
        for v in answer_mapping.values()
    )
)

assert len(answer_mapping) == 50

print("50 questions × 5 ratings = 250 mappings")

ANSWER MAPPING
Questions mapped : 50
Answers mapped   : 250
50 questions × 5 ratings = 250 mappings


In [9]:
# ============================================================
# CELL 9 — RAW → SCORED DATA FUNCTION
# ============================================================

def process_assessment(
    df,
    question_columns,
    rater_type
):

    results = []

    for row_index, row in df.iterrows():

        # ----------------------------------------------------
        # EMPLOYEE NAME
        # ----------------------------------------------------

        if rater_type == "Self":

            employee_name = row.get(
                "Nama",
                ""
            )

        else:

            employee_name = row.get(
                "Nama bawahan yang dinilai",
                ""
            )

        employee_name = (
            str(employee_name).strip()
            if not pd.isna(employee_name)
            else ""
        )

        # ----------------------------------------------------
        # OTHER METADATA
        # ----------------------------------------------------

        email = row.get(
            "Email",
            ""
        )

        completion_time = row.get(
            "Completion time",
            None
        )

        last_modified_time = row.get(
            "Last modified time",
            None
        )

        # ----------------------------------------------------
        # QUESTIONS
        # ----------------------------------------------------

        for question_col in question_columns:

            question_normalized = (
                normalize_text(question_col)
            )

            answer = row.get(
                question_col,
                ""
            )

            if pd.isna(answer):
                continue

            answer = str(answer).strip()

            if not answer:
                continue

            answer_normalized = (
                normalize_text(answer)
            )

            # ------------------------------------------------
            # FIND MASTER
            # ------------------------------------------------

            master_rows = indicator_master_df[
                indicator_master_df[
                    "question_normalized"
                ]
                == question_normalized
            ]

            if master_rows.empty:

                results.append({

                    "rater_type": rater_type,

                    "employee_name": employee_name,

                    "email": email,

                    "question": question_col,

                    "answer_text": answer,

                    "rating": np.nan,

                    "mapping_status": "QUESTION_NOT_FOUND",

                    "indicator_id": "",

                    "dimension": "",

                    "indicator_no": np.nan,

                    "indicator": "",
                })

                continue

            master_row = master_rows.iloc[0]

            # ------------------------------------------------
            # MATCH ANSWER TO LEVEL 1-5
            # ------------------------------------------------

            rating = answer_mapping[
                question_normalized
            ].get(
                answer_normalized,
                np.nan
            )

            if pd.isna(rating):

                mapping_status = (
                    "ANSWER_NOT_FOUND"
                )

            else:

                mapping_status = "MATCHED"

            # ------------------------------------------------
            # APPEND
            # ------------------------------------------------

            results.append({

                "rater_type": rater_type,

                "employee_name": employee_name,

                "email": email,

                "completion_time":
                    completion_time,

                "last_modified_time":
                    last_modified_time,

                "question": question_col,

                "answer_text": answer,

                "rating": rating,

                "mapping_status":
                    mapping_status,

                "indicator_id":
                    master_row["indicator_id"],

                "dimension":
                    master_row["dimension"],

                "indicator_no":
                    master_row["indicator_no"],

                "indicator":
                    master_row["indicator"],

            })

    return pd.DataFrame(results)

In [10]:
# ============================================================
# CELL 10 — PROCESS SELF
# ============================================================

self_scored_df = process_assessment(
    df=df_self,
    question_columns=self_question_cols,
    rater_type="Self"
)

print("========================================")
print("SELF ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(self_scored_df)
)

print(
    "\nMapping status:"
)

print(
    self_scored_df[
        "mapping_status"
    ].value_counts()
)

SELF ASSESSMENT
Total records : 1600

Mapping status:
mapping_status
MATCHED    1600
Name: count, dtype: int64


In [11]:
# ============================================================
# CELL 11 — PROCESS SUPERIOR
# ============================================================

superior_scored_df = process_assessment(
    df=df_superior,
    question_columns=superior_question_cols,
    rater_type="Superior"
)

print("========================================")
print("SUPERIOR ASSESSMENT")
print("========================================")

print(
    "Total records :",
    len(superior_scored_df)
)

print(
    "\nMapping status:"
)

print(
    superior_scored_df[
        "mapping_status"
    ].value_counts()
)

SUPERIOR ASSESSMENT
Total records : 1500

Mapping status:
mapping_status
MATCHED    1500
Name: count, dtype: int64


In [12]:
# ============================================================
# CELL 12 — COMBINE + VALIDATE SCORING
# ============================================================

scored_answers_df = pd.concat(
    [
        self_scored_df,
        superior_scored_df
    ],
    ignore_index=True
)

total_records = len(
    scored_answers_df
)

matched_records = (
    scored_answers_df[
        scored_answers_df[
            "mapping_status"
        ] == "MATCHED"
    ]
    .shape[0]
)

unmatched_records = (
    total_records
    - matched_records
)

coverage = (
    matched_records / total_records
    if total_records > 0
    else 0
)

print("========================================")
print("SCORING VALIDATION")
print("========================================")

print("Total records :", total_records)
print("Matched       :", matched_records)
print("Unmatched     :", unmatched_records)
print(
    "Coverage      :",
    f"{coverage:.2%}"
)

# ------------------------------------------------------------
# IMPORTANT
# ------------------------------------------------------------

if unmatched_records > 0:

    print(
        "\nWARNING: Ada jawaban yang tidak cocok."
    )

    display(
        scored_answers_df[
            scored_answers_df[
                "mapping_status"
            ] != "MATCHED"
        ][
            [
                "rater_type",
                "employee_name",
                "question",
                "answer_text",
                "mapping_status"
            ]
        ].head(50)
    )

else:

    print(
        "\n100% answer mapping berhasil."
    )

SCORING VALIDATION
Total records : 3100
Matched       : 3100
Unmatched     : 0
Coverage      : 100.00%

100% answer mapping berhasil.


In [13]:
# ============================================================
# CELL 13 — INDICATOR SCORE
# ============================================================

indicator_score_df = (
    scored_answers_df[
        scored_answers_df[
            "mapping_status"
        ] == "MATCHED"
    ]
    .groupby(
        [
            "rater_type",
            "employee_name",
            "indicator_id"
        ],
        as_index=False
    )
    ["rating"]
    .mean()
    .rename(
        columns={
            "rating": "indicator_score"
        }
    )
)

indicator_score_df = (
    indicator_score_df.merge(
        indicator_master_df[
            [
                "indicator_id",
                "dimension",
                "indicator_no",
                "indicator"
            ]
        ],
        on="indicator_id",
        how="left"
    )
)

print("========================================")
print("INDICATOR SCORE")
print("========================================")

print(
    indicator_score_df.head(20)
)

INDICATOR SCORE
   rater_type    employee_name       indicator_id  indicator_score  \
0        Self  AGUNG DWI CAHYO  change_agility_01              1.0   
1        Self  AGUNG DWI CAHYO  change_agility_02              2.0   
2        Self  AGUNG DWI CAHYO  change_agility_03              3.0   
3        Self  AGUNG DWI CAHYO  change_agility_04              4.0   
4        Self  AGUNG DWI CAHYO  change_agility_05              5.0   
5        Self  AGUNG DWI CAHYO  change_agility_06              1.0   
6        Self  AGUNG DWI CAHYO  change_agility_07              2.0   
7        Self  AGUNG DWI CAHYO  change_agility_08              3.0   
8        Self  AGUNG DWI CAHYO  change_agility_09              4.0   
9        Self  AGUNG DWI CAHYO  change_agility_10              5.0   
10       Self  AGUNG DWI CAHYO  mental_agility_01              1.0   
11       Self  AGUNG DWI CAHYO  mental_agility_02              2.0   
12       Self  AGUNG DWI CAHYO  mental_agility_03              3.0   
13  

In [14]:
# ============================================================
# CELL 14 — 5 DIMENSION SCORE
# ============================================================

dimension_score_df = (
    indicator_score_df
    .groupby(
        [
            "rater_type",
            "employee_name",
            "dimension"
        ],
        as_index=False
    )
    ["indicator_score"]
    .mean()
    .rename(
        columns={
            "indicator_score":
                "dimension_score"
        }
    )
)

print("========================================")
print("DIMENSION SCORE")
print("========================================")

display(
    dimension_score_df
)

DIMENSION SCORE


,rater_type,employee_name,dimension,dimension_score
0,Self,AGUNG DWI CAHYO,change_agility,3.0
1,Self,AGUNG DWI CAHYO,mental_agility,3.0
2,Self,AGUNG DWI CAHYO,people_agility,3.0
3,Self,AGUNG DWI CAHYO,result_agility,3.0
4,Self,AGUNG DWI CAHYO,self_awareness,3.0
...,...,...,...,...
305,Superior,Veronika Geuthredha Anastasia Noya,change_agility,3.0
306,Superior,Veronika Geuthredha Anastasia Noya,mental_agility,3.0
307,Superior,Veronika Geuthredha Anastasia Noya,people_agility,3.0
308,Superior,Veronika Geuthredha Anastasia Noya,result_agility,3.0


In [15]:
# ============================================================
# CELL 15 — SELF VS SUPERIOR GAP
# ============================================================

dimension_gap_df = (
    dimension_score_df
    .pivot_table(
        index=[
            "employee_name",
            "dimension"
        ],
        columns="rater_type",
        values="dimension_score"
    )
    .reset_index()
)

# ------------------------------------------------------------
# ENSURE COLUMNS
# ------------------------------------------------------------

if "Self" not in dimension_gap_df.columns:
    dimension_gap_df["Self"] = np.nan

if "Superior" not in dimension_gap_df.columns:
    dimension_gap_df["Superior"] = np.nan

# ------------------------------------------------------------
# GAP
# ------------------------------------------------------------

dimension_gap_df["gap"] = (
    dimension_gap_df["Self"]
    -
    dimension_gap_df["Superior"]
)

print("========================================")
print("DIMENSION GAP")
print("========================================")

display(
    dimension_gap_df
)

DIMENSION GAP


rater_type,employee_name,dimension,Self,Superior,gap
0,AGUNG DWI CAHYO,change_agility,3.0,NaN,NaN
1,AGUNG DWI CAHYO,mental_agility,3.0,NaN,NaN
2,AGUNG DWI CAHYO,people_agility,3.0,NaN,NaN
3,AGUNG DWI CAHYO,result_agility,3.0,NaN,NaN
4,AGUNG DWI CAHYO,self_awareness,3.0,NaN,NaN
...,...,...,...,...,...
195,fr,change_agility,3.0,NaN,NaN
196,fr,mental_agility,3.0,NaN,NaN
197,fr,people_agility,3.0,NaN,NaN
198,fr,result_agility,3.0,NaN,NaN


In [16]:
# ============================================================
# CELL 16 — GAP CATEGORY + INTERPRETATION
# ============================================================

def classify_dimension_gap(gap):

    if pd.isna(gap):
        return ""

    if gap >= 1.00:
        return "Major Positive Gap"

    elif gap >= 0.50:
        return "Minor Positive Gap"

    elif gap > -0.50:
        return "Aligned"

    elif gap > -1.00:
        return "Minor Negative Gap"

    else:
        return "Major Negative Gap"


def gap_description(category):

    descriptions = {

        "Aligned":
            "Persepsi diri selaras dengan penilaian atasan.",

        "Minor Positive Gap":
            "Self-assessment sedikit lebih tinggi dari superior assessment.",

        "Major Positive Gap":
            "Self-assessment jauh lebih tinggi dari perilaku yang tampak menurut atasan.",

        "Minor Negative Gap":
            "Self-assessment sedikit lebih rendah dari superior assessment.",

        "Major Negative Gap":
            "Self-assessment jauh lebih rendah dari perilaku yang tampak menurut atasan.",
    }

    return descriptions.get(
        category,
        ""
    )


def gap_interpretation(category):

    interpretations = {

        "Aligned":
            "Persepsi diri selaras dengan perilaku yang ditunjukkan.",

        "Minor Positive Gap":
            "Individu merasa sudah sangat baik, tetapi atasan melihat baru cukup baik.",

        "Major Positive Gap":
            "Ada potensi blind spot; perilaku belum terlihat sekuat persepsi diri.",

        "Minor Negative Gap":
            "Individu mungkin kurang percaya diri atau belum menyadari kekuatannya.",

        "Major Negative Gap":
            "Ada hidden strength; individu tidak menyadari bahwa perilakunya sudah terlihat kuat.",
    }

    return interpretations.get(
        category,
        ""
    )


dimension_gap_df["gap_category"] = (
    dimension_gap_df["gap"]
    .apply(classify_dimension_gap)
)

dimension_gap_df["gap_description"] = (
    dimension_gap_df["gap_category"]
    .apply(gap_description)
)

dimension_gap_df["interpretation"] = (
    dimension_gap_df["gap_category"]
    .apply(gap_interpretation)
)

display(
    dimension_gap_df
)

rater_type,employee_name,dimension,Self,Superior,gap,gap_category,gap_description,interpretation
0,AGUNG DWI CAHYO,change_agility,3.0,NaN,NaN,,,
1,AGUNG DWI CAHYO,mental_agility,3.0,NaN,NaN,,,
2,AGUNG DWI CAHYO,people_agility,3.0,NaN,NaN,,,
3,AGUNG DWI CAHYO,result_agility,3.0,NaN,NaN,,,
4,AGUNG DWI CAHYO,self_awareness,3.0,NaN,NaN,,,
...,...,...,...,...,...,...,...,...
195,fr,change_agility,3.0,NaN,NaN,,,
196,fr,mental_agility,3.0,NaN,NaN,,,
197,fr,people_agility,3.0,NaN,NaN,,,
198,fr,result_agility,3.0,NaN,NaN,,,


In [17]:
# ============================================================
# CELL 17 — FUNCTION FILL MASTER
# ============================================================

def fill_master_for_employee(
    employee_name,
    master_file,
    scored_answers,
    dimension_scores
):

    # --------------------------------------------------------
    # LOAD MASTER TEMPLATE
    # --------------------------------------------------------

    wb = load_workbook(
        master_file,
        data_only=False
    )

    # --------------------------------------------------------
    # FILTER DATA FOR SELECTED EMPLOYEE
    # --------------------------------------------------------

    employee_data = scored_answers[
        scored_answers["employee_name"] == employee_name
    ].copy()

    employee_dimension = dimension_scores[
        dimension_scores["employee_name"] == employee_name
    ].copy()

    # --------------------------------------------------------
    # HELPER — FORMAT SCORE
    # --------------------------------------------------------

    def apply_score_format(cell, value):

        if value is None or pd.isna(value):
            cell.value = None
            return

        # Tetap numeric
        numeric_value = float(value)

        cell.value = numeric_value

        # ----------------------------------------------------
        # FORMAT:
        #
        # 4       → 4
        # 4.1     → 4.10
        # 4.2     → 4.20
        # 4.25    → 4.25
        # 4.5     → 4.50
        # 4.125   → 4.13
        # -0.5    → -0.50
        # 0       → 0
        # ----------------------------------------------------

        rounded_value = round(
            numeric_value,
            2
        )

        if rounded_value.is_integer():

            cell.number_format = "0"

        else:

            cell.number_format = "0.00"

    # --------------------------------------------------------
    # PROCESS EACH OF 5 DIMENSIONS
    # --------------------------------------------------------

    for dimension_key, sheet_name in DIMENSIONS.items():

        ws = wb[sheet_name]

        # ----------------------------------------------------
        # FILTER EMPLOYEE DATA BY DIMENSION
        # ----------------------------------------------------

        dim_answers = employee_data[
            employee_data["dimension"] == dimension_key
        ].copy()

        dim_score = employee_dimension[
            employee_dimension["dimension"] == dimension_key
        ].copy()

        # ====================================================
        # FILL 10 INDICATORS
        #
        # Row 13 = Indicator 01
        # Row 14 = Indicator 02
        # ...
        # Row 22 = Indicator 10
        #
        # Jangan membaca Column A karena berisi formula.
        # ====================================================

        for indicator_no, row in enumerate(
            range(13, 23),
            start=1
        ):

            indicator_id = (
                f"{dimension_key}_{indicator_no:02d}"
            )

            indicator_data = dim_answers[
                dim_answers["indicator_id"] == indicator_id
            ].copy()

            # =================================================
            # SELF SCORE → COLUMN I
            # =================================================

            self_data = indicator_data[
                indicator_data["rater_type"] == "Self"
            ]

            if not self_data.empty:

                self_rating = pd.to_numeric(
                    self_data["rating"],
                    errors="coerce"
                ).mean()

                apply_score_format(
                    ws.cell(
                        row=row,
                        column=9
                    ),
                    self_rating
                )

            else:

                ws.cell(
                    row=row,
                    column=9
                ).value = None

            # =================================================
            # SUPERIOR SCORE → COLUMN K
            # =================================================

            superior_data = indicator_data[
                indicator_data["rater_type"] == "Superior"
            ]

            if not superior_data.empty:

                superior_rating = pd.to_numeric(
                    superior_data["rating"],
                    errors="coerce"
                ).mean()

                apply_score_format(
                    ws.cell(
                        row=row,
                        column=11
                    ),
                    superior_rating
                )

            else:

                ws.cell(
                    row=row,
                    column=11
                ).value = None

            # =================================================
            # GET SELF + SUPERIOR VALUE
            # =================================================

            self_value = ws.cell(
                row=row,
                column=9
            ).value

            superior_value = ws.cell(
                row=row,
                column=11
            ).value

            # =================================================
            # GAP SCORE → COLUMN M
            # GAP CATEGORY → COLUMN N
            # =================================================

            if (
                self_value is not None
                and superior_value is not None
            ):

                gap = (
                    float(self_value)
                    -
                    float(superior_value)
                )

                # ------------------------------------------------
                # GAP SCORE
                # ------------------------------------------------

                apply_score_format(
                    ws.cell(
                        row=row,
                        column=13
                    ),
                    gap
                )

                # ------------------------------------------------
                # INDICATOR GAP CATEGORY
                # ------------------------------------------------

                if gap >= 2:

                    category = (
                        "Major Positive Gap"
                    )

                elif gap >= 1:

                    category = (
                        "Minor Positive Gap"
                    )

                elif gap == 0:

                    category = "No Gap"

                elif gap >= -1:

                    category = (
                        "Minor Negative Gap"
                    )

                else:

                    category = (
                        "Major Negative Gap"
                    )

                ws.cell(
                    row=row,
                    column=14
                ).value = category

            else:

                ws.cell(
                    row=row,
                    column=13
                ).value = None

                ws.cell(
                    row=row,
                    column=14
                ).value = None

        # =====================================================
        # DIMENSION SUMMARY — ROW 23
        # =====================================================

        if not dim_score.empty:

            self_score = dim_score.iloc[0].get(
                "Self",
                np.nan
            )

            superior_score = dim_score.iloc[0].get(
                "Superior",
                np.nan
            )

            # -------------------------------------------------
            # CONVERT TO NUMERIC
            # -------------------------------------------------

            self_score = pd.to_numeric(
                self_score,
                errors="coerce"
            )

            superior_score = pd.to_numeric(
                superior_score,
                errors="coerce"
            )

            # -------------------------------------------------
            # SELF AVERAGE → I23
            # -------------------------------------------------

            apply_score_format(
                ws["I23"],
                self_score
            )

            # -------------------------------------------------
            # SUPERIOR AVERAGE → K23
            # -------------------------------------------------

            apply_score_format(
                ws["K23"],
                superior_score
            )

            # -------------------------------------------------
            # GAP → M23
            # -------------------------------------------------

            if (
                not pd.isna(self_score)
                and
                not pd.isna(superior_score)
            ):

                gap = (
                    float(self_score)
                    -
                    float(superior_score)
                )

                apply_score_format(
                    ws["M23"],
                    gap
                )

                # ------------------------------------------------
                # DIMENSION GAP CATEGORY
                # ------------------------------------------------

                category = classify_dimension_gap(
                    gap
                )

                ws["N23"] = category

                # ------------------------------------------------
                # DESCRIPTION
                # ------------------------------------------------

                ws["O23"] = gap_description(
                    category
                )

                # ------------------------------------------------
                # INTERPRETATION
                # ------------------------------------------------

                ws["P23"] = gap_interpretation(
                    category
                )

            else:

                ws["M23"] = None
                ws["N23"] = None
                ws["O23"] = None
                ws["P23"] = None

        else:

            # -------------------------------------------------
            # NO DIMENSION SCORE
            # -------------------------------------------------

            ws["I23"] = None
            ws["K23"] = None
            ws["M23"] = None
            ws["N23"] = None
            ws["O23"] = None
            ws["P23"] = None

    # --------------------------------------------------------
    # RETURN WORKBOOK
    #
    # IMPORTANT:
    # Cell 17 TIDAK SAVE FILE.
    # Cell 18 yang melakukan save.
    # --------------------------------------------------------

    return wb

In [18]:
# ============================================================
# CELL 18 — TEST GENERATE MASTER FOR ONE EMPLOYEE
# ============================================================

# ------------------------------------------------------------
# PREPARE DIMENSION SCORE
# ------------------------------------------------------------

dimension_pivot = (
    dimension_score_df
    .pivot_table(
        index=[
            "employee_name",
            "dimension"
        ],
        columns="rater_type",
        values="dimension_score"
    )
    .reset_index()
)

# ------------------------------------------------------------
# PASTIKAN KOLOM SELF DAN SUPERIOR ADA
# ------------------------------------------------------------

if "Self" not in dimension_pivot.columns:

    dimension_pivot["Self"] = np.nan

if "Superior" not in dimension_pivot.columns:

    dimension_pivot["Superior"] = np.nan


# ------------------------------------------------------------
# GET EMPLOYEE LIST
# ------------------------------------------------------------

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
)


# ------------------------------------------------------------
# DISPLAY EMPLOYEE
# ------------------------------------------------------------

print("============================================")
print("EMPLOYEE YANG TERDETEKSI")
print("============================================")

for employee in employees:
    print("-", employee)

print()
print("Total employee:", len(employees))


# ------------------------------------------------------------
# STOP IF NO EMPLOYEE
# ------------------------------------------------------------

if len(employees) == 0:

    raise ValueError(
        "Tidak ada employee dari Self Assessment."
    )


# ------------------------------------------------------------
# TEST EMPLOYEE PERTAMA
# ------------------------------------------------------------

test_employee = employees[0]

print()
print("============================================")
print("TEST EMPLOYEE")
print("============================================")
print("Employee:", test_employee)


# ------------------------------------------------------------
# CREATE TEST OUTPUT PATH
# ------------------------------------------------------------

safe_name = re.sub(
    r'[\\/*?:"<>|]',
    "_",
    test_employee
).strip()

test_output_file = (
    OUTPUT_DIR
    / f"TEST_Learning_Agility_{safe_name}.xlsx"
)


# ------------------------------------------------------------
# FILL MASTER
# ------------------------------------------------------------

test_wb = fill_master_for_employee(
    employee_name=test_employee,
    master_file=MASTER_FILE,
    scored_answers=scored_answers_df,
    dimension_scores=dimension_pivot
)


# ------------------------------------------------------------
# SAVE TEST FILE
# ------------------------------------------------------------

test_wb.save(
    test_output_file
)


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print()
print("============================================")
print("TEST GENERATION SELESAI")
print("============================================")

print(
    "Output:",
    test_output_file
)

print(
    "Exists:",
    test_output_file.exists()
)

EMPLOYEE YANG TERDETEKSI
- AGUNG DWI CAHYO
- Ade Rizqie Fonna
- Agung Sedayu
- Ana Kholifah
- Anggit Catur Nugroho
- Charli Sanjuan Siahaan
- Deddie Yunawan Wicaksono Drajat
- Dedek Saputra
- Diah Puspita
- Elisabeth Riviany Dosi
- Elrangga Leonardo Falloan
- FITRIA NOER AZIZAH
- Febrian Dwi Christian
- Febrika Rouly Lbn Raja
- Filsafah Triharyanti
- Julia Permata Edwita
- Kharisma Wulandhari
- Linda Puspitasari
- Lusi Prildayanti
- Muhammad Rifqi Nur Hadi
- Nanik Khalimatus
- Nisrina Guzmarani
- Rachma Deyana
- Raditya Adi Nugroho
- Rizqi Muharromah
- Samuel Eko Yulianto
- Steven William Yudansha
- Suthan Rizky Fadhlurrahman
- Syamsul Najib Khoirullah
- Ujang Yusup Sulaeman
- Veronika Noya
- fr

Total employee: 32

TEST EMPLOYEE
Employee: AGUNG DWI CAHYO


c:\Users\zerox\OneDrive\Documents\Desktop\learning\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)



TEST GENERATION SELESAI
Output: C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\processed\master_output\TEST_Learning_Agility_AGUNG DWI CAHYO.xlsx
Exists: True


In [19]:
# ============================================================
# CELL 19 — VALIDASI HASIL MASTER
# ============================================================

def validate_master_output(output_file):

    wb = load_workbook(
        output_file,
        data_only=False
    )

    validation = []

    for dimension_key, sheet_name in DIMENSIONS.items():

        ws = wb[sheet_name]

        # ----------------------------------------------------
        # HITUNG JUMLAH RATING
        # ----------------------------------------------------

        self_count = sum(
            ws.cell(row=row, column=9).value is not None
            for row in range(13, 23)
        )

        superior_count = sum(
            ws.cell(row=row, column=11).value is not None
            for row in range(13, 23)
        )

        gap_count = sum(
            ws.cell(row=row, column=13).value is not None
            for row in range(13, 23)
        )

        # ----------------------------------------------------
        # AMBIL SUMMARY
        # ----------------------------------------------------

        self_average = ws["I23"].value
        superior_average = ws["K23"].value
        dimension_gap = ws["M23"].value
        gap_category = ws["N23"].value

        # ----------------------------------------------------
        # SIMPAN HASIL VALIDASI
        # ----------------------------------------------------

        validation.append({
            "dimension": dimension_key,
            "sheet": sheet_name,
            "self_ratings": self_count,
            "superior_ratings": superior_count,
            "gaps": gap_count,
            "self_average": self_average,
            "superior_average": superior_average,
            "dimension_gap": dimension_gap,
            "gap_category": gap_category
        })

    return pd.DataFrame(validation)

In [20]:
# ============================================================
# CELL 19B — TEST VALIDASI
# ============================================================

if test_output_file.exists():

    print("Testing file:")
    print(test_output_file)
    print()

    validation_df = validate_master_output(
        test_output_file
    )

    display(validation_df)

else:

    print("File test belum ditemukan.")
    print("Pastikan CELL 18 sudah berhasil dijalankan.")

Testing file:
C:\Users\zerox\OneDrive\Documents\Desktop\learning\data\processed\master_output\TEST_Learning_Agility_AGUNG DWI CAHYO.xlsx



,dimension,sheet,self_ratings,superior_ratings,gaps,self_average,superior_average,dimension_gap,gap_category
0,mental_agility,mental agility,10,0,0,3,None,None,None
1,people_agility,people agility,10,0,0,3,None,None,None
2,change_agility,change agility,10,0,0,3,None,None,None
3,result_agility,result agility,10,0,0,3,None,None,None
4,self_awareness,self awareness,10,0,0,3,None,None,None


In [21]:
# ============================================================
# CELL 20 — FUNCTION TO FILL PROFILE SUMMARY
# ============================================================

def fill_profile_summary(
    wb,
    employee_name
):

    ws = wb["Profile Summary"]

    # ========================================================
    # DIMENSION CONFIGURATION
    # ========================================================

    dimension_rows = {
        "mental agility": 14,
        "people agility": 15,
        "change agility": 16,
        "result agility": 17,
        "self awareness": 18
    }

    assessment_sheets = {
        "mental agility": "mental agility",
        "people agility": "people agility",
        "change agility": "change agility",
        "result agility": "result agility",
        "self awareness": "self awareness"
    }

    # ========================================================
    # EMPLOYEE NAME
    # ========================================================
    # Profile Summary mengambil identitas dari sheet Identitas.
    # Tidak perlu mengisi ulang nama di sini.
    # ========================================================

    # ========================================================
    # HELPER — FORMAT SCORE
    # ========================================================

    def apply_score_format(cell, value):

        if value is None or pd.isna(value):
            cell.value = None
            return

        numeric_value = float(value)

        cell.value = numeric_value

        rounded_value = round(
            numeric_value,
            2
        )

        if rounded_value.is_integer():

            cell.number_format = "0"

        else:

            cell.number_format = "0.00"

    # ========================================================
    # FUNCTION — PERFORMANCE LEVEL
    # ========================================================

    def performance_level(score):

        if score is None:
            return ""

        if pd.isna(score):
            return ""

        score = float(score)

        if score >= 4.5:

            return "Excellent"

        elif score >= 4.0:

            return "Strong"

        elif score >= 3.0:

            return "Meet"

        else:

            return "Needs Development"

    # ========================================================
    # FUNCTION — GET INDICATORS BY SUPERIOR SCORE
    # ========================================================

    def get_indicators_by_score(
        assessment_ws,
        target_score
    ):

        indicators = []

        for row in range(13, 23):

            # ------------------------------------------------
            # INDICATOR
            # ------------------------------------------------

            indicator = assessment_ws.cell(
                row=row,
                column=2
            ).value

            if indicator is None:
                continue

            # ------------------------------------------------
            # SUPERIOR SCORE
            # ------------------------------------------------

            superior_score = assessment_ws.cell(
                row=row,
                column=11
            ).value

            if superior_score is None:
                continue

            # ------------------------------------------------
            # CONVERT SCORE
            # ------------------------------------------------

            try:

                superior_score = float(
                    superior_score
                )

            except (ValueError, TypeError):

                continue

            # ------------------------------------------------
            # MATCH SCORE
            # ------------------------------------------------

            if superior_score == target_score:

                indicators.append(
                    str(indicator).strip()
                )

        # ----------------------------------------------------
        # NO INDICATOR
        # ----------------------------------------------------

        if not indicators:
            return ""

        # ----------------------------------------------------
        # FORMAT LIST
        # ----------------------------------------------------

        return "\n".join(
            f"• {indicator}"
            for indicator in indicators
        )

    # ========================================================
    # PROCESS 5 DIMENSIONS
    # ========================================================

    dimension_scores = []

    for dimension_name, summary_row in dimension_rows.items():

        sheet_name = assessment_sheets[
            dimension_name
        ]

        assessment_ws = wb[sheet_name]

        # ====================================================
        # SUPERIOR DIMENSION SCORE
        # ====================================================

        superior_score = assessment_ws["K23"].value

        if superior_score is not None:

            try:

                superior_score = float(
                    superior_score
                )

            except (ValueError, TypeError):

                superior_score = None

        # ====================================================
        # COLUMN B — AVERAGE SCORE
        # ====================================================

        apply_score_format(
            ws.cell(
                row=summary_row,
                column=2
            ),
            superior_score
        )

        # ====================================================
        # COLUMN C — PERFORMANCE LEVEL
        # ====================================================

        ws.cell(
            row=summary_row,
            column=3
        ).value = performance_level(
            superior_score
        )

        # ====================================================
        # COLUMN D — SCORE 5
        # Outstanding Strength
        # ====================================================

        ws.cell(
            row=summary_row,
            column=4
        ).value = get_indicators_by_score(
            assessment_ws,
            5
        )

        # ====================================================
        # COLUMN E — SCORE 4
        # Key Strength
        # ====================================================

        ws.cell(
            row=summary_row,
            column=5
        ).value = get_indicators_by_score(
            assessment_ws,
            4
        )

        # ====================================================
        # COLUMN F — SCORE 3
        # Developing Areas
        # ====================================================

        ws.cell(
            row=summary_row,
            column=6
        ).value = get_indicators_by_score(
            assessment_ws,
            3
        )

        # ====================================================
        # COLUMN G — SCORE 2
        # Focus for Improvement
        # ====================================================

        ws.cell(
            row=summary_row,
            column=7
        ).value = get_indicators_by_score(
            assessment_ws,
            2
        )

        # ====================================================
        # COLUMN H — SCORE 1
        # Priority for Development
        # ====================================================

        ws.cell(
            row=summary_row,
            column=8
        ).value = get_indicators_by_score(
            assessment_ws,
            1
        )

        # ----------------------------------------------------
        # SAVE SCORE FOR OVERALL
        # ----------------------------------------------------

        if superior_score is not None:

            dimension_scores.append(
                superior_score
            )

    # ========================================================
    # OVERALL
    # ========================================================

    if dimension_scores:

        overall_score = (
            sum(dimension_scores)
            /
            len(dimension_scores)
        )

        # ----------------------------------------------------
        # OVERALL SCORE
        # ----------------------------------------------------

        apply_score_format(
            ws["B19"],
            overall_score
        )

        # ----------------------------------------------------
        # OVERALL PERFORMANCE
        # ----------------------------------------------------

        if overall_score >= 4.5:

            overall_level = "Highly Agile"

        elif overall_score >= 4.0:

            overall_level = "Agile"

        elif overall_score >= 3.0:

            overall_level = "Moderately Agile"

        else:

            overall_level = "Developing"

        ws["C19"] = overall_level

    else:

        ws["B19"] = None
        ws["C19"] = ""

    # ========================================================
    # RETURN WORKBOOK
    # ========================================================

    return wb

In [22]:
# ============================================================
# CELL 21 — FILL PROFILE SUMMARY GAP DETAIL
# ============================================================

def fill_profile_summary_gap(wb):

    ws_profile = wb["Profile Summary"]

    # ========================================================
    # HELPER — FORMAT SCORE
    # ========================================================

    def apply_score_format(cell, value):

        if value is None or pd.isna(value):
            cell.value = None
            return

        numeric_value = float(value)

        cell.value = numeric_value

        rounded_value = round(
            numeric_value,
            2
        )

        if rounded_value.is_integer():

            cell.number_format = "0"

        else:

            cell.number_format = "0.00"

    # ========================================================
    # KONFIGURASI SECTION PROFILE SUMMARY
    # ========================================================

    sections = [
        {
            "dimension": "mental_agility",
            "start_row": 25,
            "average_row": 35
        },
        {
            "dimension": "people_agility",
            "start_row": 39,
            "average_row": 49
        },
        {
            "dimension": "change_agility",
            "start_row": 53,
            "average_row": 63
        },
        {
            "dimension": "result_agility",
            "start_row": 68,
            "average_row": 78
        },
        {
            "dimension": "self_awareness",
            "start_row": 83,
            "average_row": 93
        }
    ]

    # ========================================================
    # LOOP SETIAP DIMENSION
    # ========================================================

    for section in sections:

        dimension_key = section["dimension"]
        start_row = section["start_row"]
        average_row = section["average_row"]

        sheet_name = DIMENSIONS[
            dimension_key
        ]

        ws_dimension = wb[sheet_name]

        # ====================================================
        # COPY 10 INDICATORS
        # ====================================================

        for i in range(10):

            source_row = 13 + i
            target_row = start_row + i

            # ------------------------------------------------
            # INDICATOR
            # ------------------------------------------------

            ws_profile.cell(
                row=target_row,
                column=1
            ).value = ws_dimension.cell(
                row=source_row,
                column=2
            ).value

            # ------------------------------------------------
            # SELF
            # ------------------------------------------------

            self_value = ws_dimension.cell(
                row=source_row,
                column=9
            ).value

            apply_score_format(
                ws_profile.cell(
                    row=target_row,
                    column=2
                ),
                self_value
            )

            # ------------------------------------------------
            # SUPERIOR
            # ------------------------------------------------

            superior_value = ws_dimension.cell(
                row=source_row,
                column=11
            ).value

            apply_score_format(
                ws_profile.cell(
                    row=target_row,
                    column=3
                ),
                superior_value
            )

            # ------------------------------------------------
            # GAP SCORE
            # ------------------------------------------------

            gap_value = ws_dimension.cell(
                row=source_row,
                column=13
            ).value

            apply_score_format(
                ws_profile.cell(
                    row=target_row,
                    column=4
                ),
                gap_value
            )

            # ------------------------------------------------
            # GAP CATEGORY
            # ------------------------------------------------

            ws_profile.cell(
                row=target_row,
                column=5
            ).value = ws_dimension.cell(
                row=source_row,
                column=14
            ).value

        # ====================================================
        # COPY DIMENSION AVERAGE
        # ====================================================

        # ----------------------------------------------------
        # SELF AVERAGE
        # ----------------------------------------------------

        self_average = ws_dimension[
            "I23"
        ].value

        apply_score_format(
            ws_profile.cell(
                row=average_row,
                column=2
            ),
            self_average
        )

        # ----------------------------------------------------
        # SUPERIOR AVERAGE
        # ----------------------------------------------------

        superior_average = ws_dimension[
            "K23"
        ].value

        apply_score_format(
            ws_profile.cell(
                row=average_row,
                column=3
            ),
            superior_average
        )

        # ----------------------------------------------------
        # GAP
        # ----------------------------------------------------

        dimension_gap = ws_dimension[
            "M23"
        ].value

        apply_score_format(
            ws_profile.cell(
                row=average_row,
                column=4
            ),
            dimension_gap
        )

        # ----------------------------------------------------
        # GAP CATEGORY
        # ----------------------------------------------------

        ws_profile.cell(
            row=average_row,
            column=5
        ).value = ws_dimension[
            "N23"
        ].value

    return wb

In [23]:
# ============================================================
# CELL 22 — FILL PROFILE SUMMARY OVERALL
# ============================================================

def fill_profile_summary_summary(wb):

    ws = wb["Profile Summary"]

    # ========================================================
    # HELPER — FORMAT SCORE
    # ========================================================

    def apply_score_format(cell, value):

        if value is None or pd.isna(value):
            cell.value = None
            return

        numeric_value = float(value)

        cell.value = numeric_value

        rounded_value = round(
            numeric_value,
            2
        )

        if rounded_value.is_integer():

            cell.number_format = "0"

        else:

            cell.number_format = "0.00"

    # ========================================================
    # SUMBER DATA AVERAGE SETIAP DIMENSION
    # ========================================================

    dimension_rows = [
        ("Mental Agility", 35),
        ("People Agility", 49),
        ("Change Agility", 63),
        ("Result Agility", 78),
        ("Self Awareness", 93)
    ]

    # ========================================================
    # COPY KE SUMMARY ROW 98–102
    # ========================================================

    target_start_row = 98

    for i, (dimension_name, source_row) in enumerate(
        dimension_rows
    ):

        target_row = target_start_row + i

        # ----------------------------------------------------
        # DIMENSION
        # ----------------------------------------------------

        ws.cell(
            row=target_row,
            column=1
        ).value = dimension_name

        # ----------------------------------------------------
        # SELF
        # ----------------------------------------------------

        self_value = ws.cell(
            row=source_row,
            column=2
        ).value

        apply_score_format(
            ws.cell(
                row=target_row,
                column=2
            ),
            self_value
        )

        # ----------------------------------------------------
        # SUPERIOR
        # ----------------------------------------------------

        superior_value = ws.cell(
            row=source_row,
            column=3
        ).value

        apply_score_format(
            ws.cell(
                row=target_row,
                column=3
            ),
            superior_value
        )

        # ----------------------------------------------------
        # GAP
        # ----------------------------------------------------

        gap_value = ws.cell(
            row=source_row,
            column=4
        ).value

        apply_score_format(
            ws.cell(
                row=target_row,
                column=4
            ),
            gap_value
        )

        # ----------------------------------------------------
        # GAP CATEGORY
        # ----------------------------------------------------

        ws.cell(
            row=target_row,
            column=5
        ).value = ws.cell(
            row=source_row,
            column=5
        ).value

    # ========================================================
    # OVERALL SELF & SUPERIOR
    # ========================================================

    self_values = []
    superior_values = []

    for row in range(98, 103):

        self_value = pd.to_numeric(
            ws.cell(
                row=row,
                column=2
            ).value,
            errors="coerce"
        )

        superior_value = pd.to_numeric(
            ws.cell(
                row=row,
                column=3
            ).value,
            errors="coerce"
        )

        if not pd.isna(self_value):

            self_values.append(
                float(self_value)
            )

        if not pd.isna(superior_value):

            superior_values.append(
                float(superior_value)
            )

    # ========================================================
    # OVERALL SELF
    # ========================================================

    if self_values:

        overall_self = np.mean(
            self_values
        )

        apply_score_format(
            ws["B103"],
            overall_self
        )

    else:

        ws["B103"] = None

    # ========================================================
    # OVERALL SUPERIOR
    # ========================================================

    if superior_values:

        overall_superior = np.mean(
            superior_values
        )

        apply_score_format(
            ws["C103"],
            overall_superior
        )

    else:

        ws["C103"] = None

    # ========================================================
    # OVERALL GAP
    # ========================================================

    if (
        ws["B103"].value is not None
        and
        ws["C103"].value is not None
    ):

        overall_gap = (
            float(ws["B103"].value)
            -
            float(ws["C103"].value)
        )

        apply_score_format(
            ws["D103"],
            overall_gap
        )

        # ----------------------------------------------------
        # OVERALL GAP CATEGORY
        # ----------------------------------------------------

        ws["E103"] = classify_dimension_gap(
            overall_gap
        )

    else:

        ws["D103"] = None
        ws["E103"] = None

    return wb

In [24]:
# ============================================================
# CELL 23A — PERFORMANCE LEVEL HELPER
# ============================================================

def performance_level_from_score(score):

    if score is None:
        return ""

    try:
        score = float(score)
    except (ValueError, TypeError):
        return ""

    if score >= 4.5:
        return "Excellent"

    elif score >= 4.0:
        return "Strong"

    elif score >= 3.0:
        return "Meet"

    else:
        return "Needs Development"

In [25]:
# ============================================================
# CELL 23 — FILL RADAR CHART DATA
# ============================================================

def fill_radar_chart_data(wb):

    ws = wb["Profile Summary"]

    # ========================================================
    # HELPER — FORMAT SCORE
    # ========================================================

    def apply_score_format(cell, value):

        if value is None or pd.isna(value):
            cell.value = None
            return

        numeric_value = float(value)

        cell.value = numeric_value

        rounded_value = round(
            numeric_value,
            2
        )

        if rounded_value.is_integer():

            cell.number_format = "0"

        else:

            cell.number_format = "0.00"

    # ========================================================
    # RADAR CONFIGURATION
    # ========================================================

    radar_config = [
        ("Mental Agility", 98, 109),
        ("People Agility", 99, 110),
        ("Change Agility", 100, 111),
        ("Result Agility", 101, 112),
        ("Self Awareness", 102, 113),
    ]

    # ========================================================
    # PROCESS RADAR DATA
    # ========================================================

    for (
        dimension_name,
        source_row,
        target_row
    ) in radar_config:

        # ----------------------------------------------------
        # DIMENSION
        # ----------------------------------------------------

        ws.cell(
            row=target_row,
            column=1
        ).value = dimension_name

        # ----------------------------------------------------
        # SCORE
        #
        # Menggunakan SUPERIOR score
        # ----------------------------------------------------

        score = ws.cell(
            row=source_row,
            column=3
        ).value

        apply_score_format(
            ws.cell(
                row=target_row,
                column=2
            ),
            score
        )

        # ----------------------------------------------------
        # PERFORMANCE LEVEL
        # ----------------------------------------------------

        ws.cell(
            row=target_row,
            column=3
        ).value = performance_level_from_score(
            score
        )

    return wb

In [26]:
# ============================================================
# CELL 24 — BUILD PROFILE SUMMARY
# ============================================================

def build_profile_summary(wb, employee_name):

    # ========================================================
    # 1. FILL DETAIL GAP
    # ========================================================

    wb = fill_profile_summary_gap(wb)

    # ========================================================
    # 2. FILL OVERALL SUMMARY
    # ========================================================

    wb = fill_profile_summary_summary(wb)

    # ========================================================
    # 3. FILL RADAR CHART DATA
    # ========================================================

    wb = fill_radar_chart_data(wb)

    return wb

In [27]:
# ============================================================
# CELL 25 — GENERATE MASTER UNTUK SEMUA KARYAWAN
# ============================================================

print("=" * 60)
print("MASTER GENERATION — SEMUA KARYAWAN")
print("=" * 60)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

generated_files = []

# ------------------------------------------------------------
# GET SEMUA EMPLOYEE DARI SELF ASSESSMENT
# ------------------------------------------------------------

employees = sorted(
    scored_answers_df[
        scored_answers_df["rater_type"] == "Self"
    ]["employee_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s != ""]
    .unique()
)

print()
print("Total employee:", len(employees))
print()

# ------------------------------------------------------------
# GENERATE SATU PER SATU
# ------------------------------------------------------------

for index, employee_name in enumerate(
    employees,
    start=1
):

    print(
        f"[{index}/{len(employees)}] "
        f"Processing: {employee_name}"
    )

    # --------------------------------------------------------
    # FILE NAME AMAN
    # --------------------------------------------------------

    safe_name = re.sub(
        r'[\\/*?:"<>|]',
        "_",
        employee_name
    ).strip()

    output_file = (
        OUTPUT_DIR
        / f"Learning_Agility_{safe_name}.xlsx"
    )

    try:

        # ----------------------------------------------------
        # LOAD MASTER TEMPLATE
        # ----------------------------------------------------

        wb = fill_master_for_employee(
            employee_name=employee_name,
            master_file=MASTER_FILE,
            scored_answers=scored_answers_df,
            dimension_scores=dimension_pivot
        )

        # ----------------------------------------------------
        # BUILD PROFILE SUMMARY
        # ----------------------------------------------------

        wb = build_profile_summary(
            wb=wb,
            employee_name=employee_name
        )

        # ----------------------------------------------------
        # SAVE
        # ----------------------------------------------------

        wb.save(output_file)

        generated_files.append(
            output_file
        )

        print(
            "    ✓ Success"
        )

    except Exception as e:

        print(
            "    ✗ ERROR:",
            repr(e)
        )

print()
print("=" * 60)
print("GENERATION SELESAI")
print("=" * 60)

print(
    "Berhasil:",
    len(generated_files),
    "/",
    len(employees)
)

print(
    "Output folder:",
    OUTPUT_DIR
)

MASTER GENERATION — SEMUA KARYAWAN

Total employee: 32

[1/32] Processing: AGUNG DWI CAHYO


c:\Users\zerox\OneDrive\Documents\Desktop\learning\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


    ✓ Success
[2/32] Processing: Ade Rizqie Fonna
    ✓ Success
[3/32] Processing: Agung Sedayu
    ✓ Success
[4/32] Processing: Ana Kholifah
    ✓ Success
[5/32] Processing: Anggit Catur Nugroho
    ✓ Success
[6/32] Processing: Charli Sanjuan Siahaan
    ✓ Success
[7/32] Processing: Deddie Yunawan Wicaksono Drajat
    ✓ Success
[8/32] Processing: Dedek Saputra
    ✓ Success
[9/32] Processing: Diah Puspita
    ✓ Success
[10/32] Processing: Elisabeth Riviany Dosi
    ✓ Success
[11/32] Processing: Elrangga Leonardo Falloan
    ✓ Success
[12/32] Processing: FITRIA NOER AZIZAH
    ✓ Success
[13/32] Processing: Febrian Dwi Christian
    ✓ Success
[14/32] Processing: Febrika Rouly Lbn Raja
    ✓ Success
[15/32] Processing: Filsafah Triharyanti
    ✓ Success
[16/32] Processing: Julia Permata Edwita
    ✓ Success
[17/32] Processing: Kharisma Wulandhari
    ✓ Success
[18/32] Processing: Linda Puspitasari
    ✓ Success
[19/32] Processing: Lusi Prildayanti
    ✓ Success
[20/32] Processing: Muhamm